# UniAssist — Week 3: Morphology, Stemming, Lemmatization and WordNet
Use the original UMT corpus to compare stemming algorithms, POS-aware lemmatization and semantic relations. We evaluate outputs on examples rather than presuming that any single algorithm is correct for every word.


In [ ]:
from pathlib import Path
import json, re, statistics
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "corpus").exists():
    ROOT = ROOT.parent
assert (ROOT / "corpus").exists(), "Run notebook from UniAssist_UMT directory"
metadata = json.loads((ROOT / "metadata.json").read_text(encoding="utf-8"))
records = metadata["documents"]
texts = [(ROOT / "corpus" / row["file"]).read_text(encoding="utf-8") for row in records]
print(len(texts), "documents loaded")


In [ ]:
import nltk, spacy
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet as wn
from nltk import pos_tag
for dataset in ("wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"):
    nltk.download(dataset, quiet=True)
porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()
nlp = spacy.blank("en")


## Task A — Compare three stemmers on vocabulary from UMT documents


In [ ]:
words = ["studies", "studying", "universities", "student", "students",
         "admissions", "admitted", "evaluation", "evaluated", "applications",
         "faculties", "scholarships", "architectural", "mobility", "programmes"]
stem_table = pd.DataFrame([{
    "word": word, "Porter": porter.stem(word),
    "Snowball": snowball.stem(word), "Lancaster": lancaster.stem(word)
} for word in words])
display(stem_table)
print("Porter distinct stems:", stem_table["Porter"].nunique())
print("Snowball distinct stems:", stem_table["Snowball"].nunique())
print("Lancaster distinct stems:", stem_table["Lancaster"].nunique())


## Task B — Corpus-level vocabulary reduction
Stemming is a string normalization technique, not guaranteed grammatical lemmatization. Compare unique terms only after lowercasing and filtering alphabetic tokens.


In [ ]:
words_in_corpus = [token.text.lower() for text in texts
                   for token in nlp.make_doc(text) if token.is_alpha]
rows = []
for name, transform in [("unchanged", lambda s:s), ("Porter", porter.stem),
                        ("Snowball", snowball.stem), ("Lancaster", lancaster.stem)]:
    forms = [transform(word) for word in words_in_corpus]
    rows.append({"method": name, "tokens": len(forms), "types": len(set(forms)),
                 "TTR": round(len(set(forms))/max(1, len(forms)), 4)})
display(pd.DataFrame(rows))


## Task C — Lemmatization without POS versus POS-aware WordNet lemmatization


In [ ]:
def wn_pos(tag):
    # Penn Treebank tag initials: adjective, verb, noun, adverb.
    return {"J":wn.ADJ, "V":wn.VERB, "N":wn.NOUN, "R":wn.ADV}.get(tag[:1], wn.NOUN)

samples = ["Students were studying admission requirements carefully.",
           "The faculties offered different programmes and scholarships.",
           "Visitors asked about better learning facilities and transferred credits."]
lemma_rows = []
for sentence in samples:
    tokens = nltk.word_tokenize(sentence, preserve_line=True)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        if word.isalpha():
            lemma_rows.append({"sentence": sentence, "word": word, "POS": tag,
                "default_noun": lemmatizer.lemmatize(word.lower()),
                "POS_aware": lemmatizer.lemmatize(word.lower(), wn_pos(tag))})
display(pd.DataFrame(lemma_rows))


## Task D — spaCy lemmatization on actual source-based text
This requires the trained English model. Compare it with the WordNet POS-aware results and note errors for UMT proper names, acronyms, and domain vocabulary.


In [ ]:
try:
    trained = spacy.load("en_core_web_sm")
except OSError:
    print("Missing spaCy language model. Run: python -m spacy download en_core_web_sm")
else:
    sample_doc = trained(texts[0][:1300])
    display(pd.DataFrame([{"token": tok.text, "lemma": tok.lemma_, "coarse_POS": tok.pos_,
                           "fine_POS": tok.tag_} for tok in sample_doc if tok.is_alpha]).head(55))


## Task E — WordNet senses, hypernyms, and semantic similarity
WordNet describes general English, not UMT's internal policies. The first WordNet sense is not automatically the correct sense in a university context.


In [ ]:
terms = ["university", "student", "course", "admission", "library", "faculty"]
for term in terms:
    senses = wn.synsets(term, pos=wn.NOUN)
    print("\n", term, "senses:", len(senses))
    for syn in senses[:2]:
        print(" ", syn.name(), "-", syn.definition())
        print("   hypernyms:", [h.name() for h in syn.hypernyms()[:3]])

pairs = [("student", "scholar"), ("university", "college"),
         ("course", "class"), ("library", "school")]
for left, right in pairs:
    a = wn.synsets(left, pos=wn.NOUN)
    b = wn.synsets(right, pos=wn.NOUN)
    value = a[0].wup_similarity(b[0]) if a and b else None
    print(left, right, "first-sense Wu-Palmer similarity:", value)


## Task F — Error analysis and implications for search


In [ ]:
examples = ["studies", "universities", "admission", "programmes",
            "Erasmus", "UMT", "better", "running", "architecture"]
display(pd.DataFrame([{"input": w, "Porter": porter.stem(w),
                       "Snowball": snowball.stem(w),
                       "noun_lemma": lemmatizer.lemmatize(w.lower())} for w in examples]))


## Submit
1. Compare 15 words across Porter, Snowball and Lancaster.
2. Explain observed vocabulary reduction and at least three overstemming/understemming cases.
3. Contrast default WordNet, POS-aware WordNet and spaCy lemmas with examples.
4. Display at least six WordNet lookup results and four semantic comparisons.
5. State which normalization you would use for the eventual TF-IDF search and why, including limitations.
